# SEIS 606: Vibe Coding
## Homework 2, Image Generation for App Specs
Dante Razo, razo3843@stthomas.edu, FA26

I've been using this GPU-accelerated notebook template since I first started at UST. It's something that I carry from class to class.

## GPU-Accelerated Environment Configuration

In [1]:
import torch

# validate CUDA setup
print("Torch CUDA Available? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Torch CUDA Version:", torch.version.cuda)
    print("Torch cuDNN Version:", torch.backends.cudnn.version())

    # print GPU information
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:", torch.cuda.get_device_name(device=i))

    # check NVIDIA driver
    !echo && nvidia-smi

# set device type
device: str = "cuda" if torch.cuda.is_available() else "cpu"

Torch CUDA Available?  True
Torch CUDA Version: 13.0
Torch cuDNN Version: 92400

GPU 0: NVIDIA GeForce RTX 5090

Fri Sep 25 01:03:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.08              KMD Version: 616.92        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
|  0%   44C    P0             64W /  460W |    3563MiB /  32607MiB |      4%      Default |
|                          

In [2]:
import gc


def free_vram() -> None:
    gc.collect()
    torch.cuda.empty_cache()


# free now, though it should be empty with a fresh kernel
free_vram()

In [3]:
import os
from pathlib import Path

# create cache location
hf_home: Path = Path("/cache/huggingface")
hf_home.mkdir(parents=True, exist_ok=True)

# set environment variables for huggingface / transformers
os.environ["HF_HOME"] = str(object=hf_home)

In [4]:
from dotenv import load_dotenv

# load environment, including HF token
load_dotenv()

False

In [5]:
# validate environment variables with assertions
assert hf_home.exists()
assert os.environ["HF_HOME"] == str(object=hf_home)

## Loading the Image Generation Model
According to my (brief) research, *Qwen Image 2.1* is the best (or among the best) open-weight text-to-image models.

In [ ]:
from diffusers.pipelines.qwenimage.pipeline_qwenimage import QwenImagePipeline
from diffusers.quantizers.pipe_quant_config import PipelineQuantizationConfig
from diffusers.quantizers.quantization_config import (
    QuantizationConfigMixin,
    QuantoConfig as DiffusersQuantoConfig,
    )
from torch import bfloat16, dtype, float32
from transformers.utils.quantization_config import (
    QuantizationConfigMixin as TransformersQuantConfigMixin,
    QuantoConfig as TransformersQuantoConfig,
    )

# define model metadata
qwen_model_id: str = "Qwen/Qwen-Image-2.1"
torch_dtype: dtype = bfloat16 if device == "cuda" else float32

# quantize the large model components to 8-bit weights
quantization_mapping: dict[str, QuantizationConfigMixin | TransformersQuantConfigMixin] = {
    "transformer": DiffusersQuantoConfig(weights_dtype="int8"),
    "text_encoder": TransformersQuantoConfig(weights="int8"),
}
quantization_config = PipelineQuantizationConfig(
    quant_mapping=quantization_mapping) # pyright: ignore[reportArgumentType]
)

# keep the quantized pipeline entirely on the selected device
pipeline: QwenImagePipeline = QwenImagePipeline.from_pretrained(
    pretrained_model_name_or_path=qwen_model_id,
    dtype=torch_dtype,
    quantization_config=quantization_config,
).to(device)

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/diffusers/quantizers/quantization_config.py:671: FutureWarning: `QuantoConfig` is deprecated and will be removed in version 1.0.0. `QuantoConfig` is deprecated and will be removed in version 1.0.0.
  deprecate("QuantoConfig", "1.0.0", deprecation_message)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
# wrapper function for generation + persisting to disk
def generate_image(prompt: str, save_path: str = "") -> None:
    image = pipeline(prompt=prompt).images[0]  # pyright: ignore[reportAttributeAccessIssue, reportCallIssue]
    image.save(save_path) if save_path else None
    display(image)

In [ ]:
generate_image(prompt="A clean UI mockup for a homelab dashboard")